# PageRank on the PHP Composer (Packagist) Dependency Graph

> **Nub Labs Playbook** — Interactive walkthrough: [nublabs.com/case-studies/pagerank](https://www.nublabs.com/case-studies/pagerank)

**The technique:** Same power iteration PageRank — different graph, same insight.  
**The dataset:** 32 top Packagist packages (curated) + optional live Packagist API fetch.  
**The real application:** Identify the highest-leverage packages in your PHP codebase — the ones a single CVE in would cascade the furthest.

**Two modes:**
1. **Curated dataset** (Cells 1–5) — runs instantly, no internet required.
2. **Live Packagist fetch** (Cells 6–8) — extends with real-time data from `packagist.org`.

**Prediction before you run it:** `psr/log` will rank #1 — not `laravel/framework` or `guzzlehttp/guzzle`.

In [ ]:
# Only needed for the live-fetch section (Cells 6–8)
!pip install requests -q

## 1. Curated Dataset — Top Packagist Packages

Verified against `packagist.org` (2024 snapshot).  
Edge `(A, B)` = *"A requires B"* in its `composer.json`.

In [ ]:
# (id, monthly_downloads_approx, description)
NODES = [
    # ── PSR interfaces — no dependencies ─────────────────────────────────────
    ("psr/log",                      500_000_000, "PSR-3 logger interface"),
    ("psr/container",                400_000_000, "PSR-11 DI container interface"),
    ("psr/http-message",             450_000_000, "PSR-7 HTTP message interface"),
    ("psr/event-dispatcher",         300_000_000, "PSR-14 event dispatcher interface"),
    ("psr/simple-cache",             250_000_000, "PSR-16 simple cache interface"),
    # ── Symfony polyfills — no dependencies ──────────────────────────────────
    ("symfony/polyfill-mbstring",    600_000_000, "mbstring polyfill"),
    ("symfony/polyfill-ctype",       500_000_000, "ctype polyfill"),
    ("symfony/polyfill-php80",       550_000_000, "PHP 8.0 polyfill"),
    ("symfony/polyfill-php81",       400_000_000, "PHP 8.1 polyfill"),
    ("symfony/polyfill-intl-normalizer", 300_000_000, "intl Normalizer polyfill"),
    # ── Other foundation packages ─────────────────────────────────────────────
    ("guzzlehttp/promises",          400_000_000, "Guzzle async promises"),
    ("ralouphie/getallheaders",      350_000_000, "getallheaders() polyfill"),
    ("doctrine/lexer",               300_000_000, "Abstract lexer for parsers"),
    ("nikic/php-parser",             250_000_000, "PHP parser library"),
    # ── Utility tier (1–2 deps) ───────────────────────────────────────────────
    ("psr/http-factory",             400_000_000, "PSR-17 HTTP factory interface"),
    ("psr/http-client",              380_000_000, "PSR-18 HTTP client interface"),
    ("symfony/polyfill-intl-idn",    280_000_000, "intl IDN polyfill"),
    ("symfony/service-contracts",    420_000_000, "Symfony service contracts"),
    ("symfony/event-dispatcher-contracts", 380_000_000, "Symfony event dispatcher contracts"),
    ("monolog/monolog",              450_000_000, "PHP logging library"),
    ("doctrine/annotations",        350_000_000, "PHP docblock annotations"),
    # ── Mid-level tier ───────────────────────────────────────────────────────
    ("symfony/event-dispatcher",     420_000_000, "Symfony event dispatcher"),
    ("symfony/console",              480_000_000, "Symfony console component"),
    ("symfony/filesystem",           380_000_000, "Symfony filesystem component"),
    ("guzzlehttp/psr7",              430_000_000, "Guzzle PSR-7 HTTP message impl"),
    # ── Higher-level tier ────────────────────────────────────────────────────
    ("symfony/config",               360_000_000, "Symfony config component"),
    ("symfony/http-foundation",      430_000_000, "Symfony HTTP foundation"),
    ("symfony/dependency-injection", 400_000_000, "Symfony DI container"),
    ("guzzlehttp/guzzle",            460_000_000, "Guzzle HTTP client"),
    # ── Application tier ─────────────────────────────────────────────────────
    ("symfony/http-kernel",          410_000_000, "Symfony HTTP kernel"),
    ("symfony/routing",              390_000_000, "Symfony routing component"),
    # ── Consumer tier ────────────────────────────────────────────────────────
    ("laravel/framework",            200_000_000, "The Laravel Framework"),
    ("phpunit/phpunit",              350_000_000, "PHP unit testing framework"),
]

# 40 directed edges: (source, target) means source requires target
EDGES = [
    # PSR interface dependencies
    ("psr/http-factory",             "psr/http-message"),
    ("psr/http-client",              "psr/http-message"),
    # Symfony polyfill chains
    ("symfony/polyfill-intl-idn",    "symfony/polyfill-intl-normalizer"),
    # Symfony contracts
    ("symfony/service-contracts",    "psr/container"),
    ("symfony/event-dispatcher-contracts", "psr/event-dispatcher"),
    # Utilities
    ("monolog/monolog",              "psr/log"),
    ("doctrine/annotations",        "doctrine/lexer"),
    # Symfony event-dispatcher
    ("symfony/event-dispatcher",     "symfony/event-dispatcher-contracts"),
    ("symfony/event-dispatcher",     "psr/event-dispatcher"),
    # Symfony console
    ("symfony/console",              "symfony/polyfill-mbstring"),
    ("symfony/console",              "symfony/polyfill-php80"),
    ("symfony/console",              "psr/log"),
    # Symfony filesystem
    ("symfony/filesystem",           "symfony/polyfill-ctype"),
    # Guzzle PSR-7
    ("guzzlehttp/psr7",              "psr/http-message"),
    ("guzzlehttp/psr7",              "psr/http-factory"),
    ("guzzlehttp/psr7",              "ralouphie/getallheaders"),
    # Symfony config
    ("symfony/config",               "symfony/filesystem"),
    # Symfony http-foundation
    ("symfony/http-foundation",      "symfony/polyfill-mbstring"),
    ("symfony/http-foundation",      "symfony/polyfill-php80"),
    # Symfony DI
    ("symfony/dependency-injection", "symfony/service-contracts"),
    ("symfony/dependency-injection", "psr/container"),
    # Guzzle
    ("guzzlehttp/guzzle",            "guzzlehttp/promises"),
    ("guzzlehttp/guzzle",            "guzzlehttp/psr7"),
    ("guzzlehttp/guzzle",            "psr/http-client"),
    ("guzzlehttp/guzzle",            "psr/http-factory"),
    # Symfony http-kernel
    ("symfony/http-kernel",          "symfony/event-dispatcher"),
    ("symfony/http-kernel",          "psr/log"),
    ("symfony/http-kernel",          "symfony/http-foundation"),
    # Symfony routing
    ("symfony/routing",              "symfony/config"),
    ("symfony/routing",              "psr/log"),
    # Laravel framework
    ("laravel/framework",            "symfony/console"),
    ("laravel/framework",            "symfony/http-foundation"),
    ("laravel/framework",            "symfony/http-kernel"),
    ("laravel/framework",            "psr/log"),
    ("laravel/framework",            "monolog/monolog"),
    ("laravel/framework",            "guzzlehttp/guzzle"),
    # PHPUnit
    ("phpunit/phpunit",              "doctrine/annotations"),
    ("phpunit/phpunit",              "symfony/console"),
    ("phpunit/phpunit",              "symfony/filesystem"),
    ("phpunit/phpunit",              "nikic/php-parser"),
]

print(f"Nodes: {len(NODES)}")
print(f"Edges: {len(EDGES)}")

## 2. PageRank Algorithm (same as npm notebook)

In [ ]:
def pagerank(node_ids, edges, damping=0.85, tolerance=1e-8, max_iter=100):
    N = len(node_ids)
    node_set = set(node_ids)

    out_adj = {n: [] for n in node_ids}
    in_adj  = {n: [] for n in node_ids}
    for src, tgt in edges:
        if src in node_set and tgt in node_set:
            out_adj[src].append(tgt)
            in_adj[tgt].append(src)

    dangling = [n for n in node_ids if not out_adj[n]]
    ranks    = {n: 1.0 / N for n in node_ids}
    history  = [dict(ranks)]
    converged_at = max_iter

    for iteration in range(max_iter):
        dangling_sum = sum(ranks[n] for n in dangling)
        new_ranks = {}
        for n in node_ids:
            link_vote = sum(
                ranks[src] / len(out_adj[src])
                for src in in_adj[n]
            )
            new_ranks[n] = (
                (1 - damping) / N
                + damping * (link_vote + dangling_sum / N)
            )
        delta = sum(abs(new_ranks[n] - ranks[n]) for n in node_ids)
        ranks = new_ranks
        history.append(dict(ranks))
        if delta < tolerance:
            converged_at = iteration + 1
            break

    return ranks, converged_at, history

## 3. Run on Curated Dataset

In [ ]:
node_ids = [n[0] for n in NODES]
ranks, converged_at, history = pagerank(node_ids, EDGES)

out_deg = {n: 0 for n in node_ids}
in_deg  = {n: 0 for n in node_ids}
for src, tgt in EDGES:
    out_deg[src] += 1
    in_deg[tgt]  += 1

max_rank     = max(ranks.values())
sorted_nodes = sorted(node_ids, key=lambda n: ranks[n], reverse=True)

print(f"Converged at iteration: {converged_at}")
print(f"Sum of all ranks      : {sum(ranks.values()):.6f}")
print()
print(f"{'#':<4} {'Package':<38} {'Rank %':>8}  {'Score':>6}  {'In':>4}  {'Out':>4}")
print("-" * 72)
for i, n in enumerate(sorted_nodes[:15], 1):
    score = ranks[n] / max_rank * 100
    print(f"{i:<4} {n:<38} {ranks[n]*100:>8.4f}%  {score:>5.1f}   {in_deg[n]:>4}   {out_deg[n]:>4}")

## 4. Visualization

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

top15 = sorted_nodes[:15]
scores = [ranks[n] / max_rank * 100 for n in top15]

def bar_color(score):
    if score >= 85: return "#F59E0B"
    if score >= 65: return "#A78BFA"
    if score >= 40: return "#60A5FA"
    if score >= 18: return "#34D399"
    return "#64748B"

colors = [bar_color(s) for s in scores]

fig, ax = plt.subplots(figsize=(12, 6))
fig.patch.set_facecolor("#0A0F1E")
ax.set_facecolor("#0A0F1E")

bars = ax.barh(range(len(top15)), scores, color=colors, height=0.7, zorder=3)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15, fontfamily="monospace", color="#94A3B8", fontsize=9)
ax.set_xlabel("Relative PageRank score (0–100)", color="#64748B", fontsize=9)
ax.set_title("PageRank — Top 15 Packagist Packages", color="white", fontsize=13, pad=14)
ax.invert_yaxis()
ax.tick_params(colors="#64748B")
ax.spines[:].set_color("#1E293B")
ax.grid(axis="x", color="#1E293B", linewidth=0.7, zorder=0)

for i, (bar, n) in enumerate(zip(bars, top15)):
    ax.text(
        bar.get_width() + 0.5, i,
        f"{ranks[n]*100:.4f}%",
        va="center", color="#64748B", fontsize=7.5, fontfamily="monospace"
    )

plt.tight_layout()
plt.show()

## 5. Why psr/log Ranks #1

The same principle as `js-tokens` in npm — **exclusive supplier to a concentrated chain**:

```
laravel/framework, symfony/http-kernel, symfony/routing, symfony/console
        │
        ▼
   psr/log  ←  also receives 100% of monolog/monolog's rank
              (monolog has only 1 outgoing edge)
```

**psr/log is a dangling node** (no outgoing edges) that collects rank from:
- `monolog/monolog` — 100% passthrough (sole outgoing edge)
- `symfony/console` — 1/3 of console's rank
- `symfony/http-kernel` — 1/3 of kernel's rank  
- `symfony/routing` — 1/2 of routing's rank
- `laravel/framework` — 1/6 of Laravel's rank

PSR interface packages (`psr/log`, `psr/http-message`) always dominate Composer graphs because they are **standards with no dependencies**, and **many independent packages implement them**.

---
## 6. Live Packagist Fetch (Optional)

The cells below extend the dataset by fetching real dependency data from `packagist.org`.  
Requires an internet connection. Rate-limited to ~1 req/sec.

In [ ]:
import requests, time

def fetch_composer_deps(package: str, version_hint: str = None) -> list[str]:
    """
    Fetch the `require` dependencies of a Packagist package.
    Returns a list of package names (excludes php/ext-* requirements).
    """
    try:
        url  = f"https://packagist.org/packages/{package}.json"
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json()["package"]

        # Pick the latest stable version
        versions = data.get("versions", {})
        stable   = [
            v for v in versions
            if not any(tag in v for tag in ("dev", "alpha", "beta", "RC"))
        ]
        if not stable:
            return []
        latest   = sorted(stable, reverse=True)[0]
        requires = versions[latest].get("require", {})

        return [
            dep for dep in requires
            if "/" in dep                    # keep only vendor/package format
            and not dep.startswith("ext-")   # exclude PHP extensions
            and dep != "php"                 # exclude PHP itself
        ]
    except Exception as e:
        print(f"  ⚠ Could not fetch {package}: {e}")
        return []

In [ ]:
# ─── Additional packages to fetch ──────────────────────────────────────────
# Add any Packagist package slug you want to include in the live graph.
EXTRA_PACKAGES = [
    "symfony/var-dumper",
    "symfony/translation",
    "symfony/string",
    "symfony/validator",
    "doctrine/orm",
    "doctrine/dbal",
    "predis/predis",
    "vlucas/phpdotenv",
    "nesbot/carbon",
]

live_edges = []
live_nodes = set(node_ids)  # start with curated nodes

print("Fetching from packagist.org …")
for pkg in EXTRA_PACKAGES:
    print(f"  {pkg} ...", end=" ")
    deps = fetch_composer_deps(pkg)
    print(f"{len(deps)} deps")
    live_nodes.add(pkg)
    for dep in deps:
        live_edges.append((pkg, dep))
        live_nodes.add(dep)
    time.sleep(0.5)  # gentle rate limiting

print(f"\nTotal nodes after live fetch: {len(live_nodes)}")
print(f"Total edges after live fetch: {len(EDGES) + len(live_edges)}")

In [ ]:
# Re-run PageRank on the extended graph
all_nodes = list(live_nodes)
all_edges = list(EDGES) + live_edges

ranks_ext, converged_ext, _ = pagerank(all_nodes, all_edges)

max_rank_ext = max(ranks_ext.values())
sorted_ext   = sorted(all_nodes, key=lambda n: ranks_ext[n], reverse=True)

print(f"Converged at iteration: {converged_ext}")
print()
print(f"{'#':<4} {'Package':<42} {'Rank %':>8}  {'Score':>6}")
print("-" * 65)
for i, n in enumerate(sorted_ext[:15], 1):
    score = ranks_ext[n] / max_rank_ext * 100
    print(f"{i:<4} {n:<42} {ranks_ext[n]*100:>8.4f}%  {score:>5.1f}")

---
## 7. Cross-Ecosystem Comparison

| Finding | npm | PHP Composer |
|---|---|---|
| **#1 package** | `js-tokens` | `psr/log` |
| **Why?** | sole dep of `loose-envify` (React ecosystem) | depended on by everything that logs; `monolog` passes 100% |
| **Framework rank** | `next` very low | `laravel/framework` very low |
| **The lesson** | Exclusivity beats popularity | Same |
| **Hidden critical nodes** | `wrappy`, `ms`, `color-name` | `psr/event-dispatcher`, `doctrine/lexer` |
| **Dangling node count** | 29 of 58 | 14 of 33 |

**The pattern holds across ecosystems:**  
Shared standards and micro-utilities that many packages converge on — with no outgoing dependencies of their own — always dominate graph-based importance metrics.